In [ ]:
import pandas as pd
import csv

In [ ]:
df = pd.read_csv('./data/weather-stations/kelowna/kelowna_stations.csv')

In [ ]:
df.sort_values(by=['Latitude'], ascending = True)

In [ ]:
total_burn_data = pd.read_csv("./data/satellite-burn/satellite_burn_cleaned.csv")
total_burn_data[:70]

In [ ]:
total_burn_data['LATITUDE'] = pd.to_numeric(total_burn_data['LATITUDE'], errors='coerce')
total_burn_data['LONGITUDE'] = pd.to_numeric(total_burn_data['LONGITUDE'], errors='coerce')

total_burn_data.dropna(subset=['LATITUDE', 'LONGITUDE'], inplace=True)

In [ ]:
def spatially_split_stations(df, num_long_splits, num_lat_splits):
    longitude_max = df['Longitude'].max()
    longitude_min = df['Longitude'].min()
    latitude_max = df['Latitude'].max()
    latitude_min = df['Latitude'].min()
    longitude_distance = longitude_max - longitude_min
    latitude_distance = latitude_max - latitude_min
    
    long_split_distance = longitude_distance / num_long_splits
    lat_split_distance = latitude_distance / num_lat_splits
    
    long_track = longitude_min
    lat_track = latitude_min
    print(f"min_long = {longitude_min}, min_lat = {latitude_min}, max_long = {longitude_max}, max_lat = {latitude_max}, long_split = {long_split_distance}, lat_split = {lat_split_distance}")
    zero_count = 0
    num_above = 0
    
    for i in range(num_lat_splits):
        lat_track = latitude_min + i * lat_split_distance
        for j in range(num_long_splits):
            long_track = longitude_min + j * long_split_distance
            
            latitude_mask_station = (df['Latitude'] < lat_track + lat_split_distance) & (df['Latitude'] >= lat_track)
            longitude_mask_station = (df['Longitude'] < long_track + long_split_distance) & (df['Longitude'] >= long_track)
            
            latitude_mask_burn = (burn_data['LATITUDE'] < lat_track + lat_split_distance) & (burn_data['LATITUDE'] >= lat_track)
            longitude_mask_burn = (burn_data['LONGITUDE'] < long_track + long_split_distance) & (burn_data['LONGITUDE'] >= long_track)
            
            station_df = df[latitude_mask_station & longitude_mask_station]
            single_burn = burn_data[latitude_mask_burn & longitude_mask_burn]['FIRE_SIZE_HA'].sum(axis=0)
            burn_array_sample.append(single_burn)
            
            print(f"total burn is {single_burn}")
            
            if single_burn >= 0.01: # in order to see number of regions with noticeable burn area
                num_above += 1
                
            if len(station_df) == 0: 
                zero_count += 1
            station_df.to_csv(f"data/weather-stations/kelowna/fine_area_{i*num_long_splits+j+1}_stations.csv")
            
            print(f"num of stations to aggregate is {len(station_df)}")
            
    print(f"zero count is {zero_count}")
    print(f"num burn above 0.01 is {num_above}")

In [ ]:
spatially_split_stations(df, 6, 6)

In [ ]:
def get_temporal_category(date_string, num_days):
    num = pd.to_datetime(date_string).dayofyear
    temporal_group = (num - 1) // num_days + 1
    return temporal_group

In [ ]:
pd.set_option('display.min_rows',300)
pd.set_option('display.max_rows',300)
longitude_max = df['Longitude'].max()
longitude_min = df['Longitude'].min()
latitude_max = df['Latitude'].max()
latitude_min = df['Latitude'].min()

my_filter = (total_burn_data['LATITUDE'] <= latitude_max) & (total_burn_data['LATITUDE'] >= latitude_min)
my_filter2 = (total_burn_data['LONGITUDE'] <= longitude_max) & (total_burn_data['LONGITUDE'] >= longitude_min)
kelowna_burn_data = total_burn_data[my_filter & my_filter2]
kelowna_burn_data[:300]

In [ ]:
import os

def spatially_split_burn(all_burn, num_long_splits, num_lat_splits, year_min, year_max, temporal_min, temporal_max, window):
    base_dir = './data/final-burn'
    
    longitude_max = all_burn['LONGITUDE'].max()
    longitude_min = all_burn['LONGITUDE'].min()
    latitude_max = all_burn['LATITUDE'].max()
    latitude_min = all_burn['LATITUDE'].min()
    longitude_distance = longitude_max - longitude_min
    latitude_distance = latitude_max - latitude_min

    long_split_distance = longitude_distance / num_long_splits
    lat_split_distance = latitude_distance / num_lat_splits

    long_track = longitude_min
    lat_track = latitude_min

    for year in range(year_min,year_max):        
        for temporal_split in range(temporal_min,temporal_max):
            segmented_burn = []
            
            burn_filter_year = all_burn['IGNITION_DATE'].str[:4] == str(year)
            burn_filter_temporal = all_burn['IGNITION_DATE'].apply(get_temporal_category, args=(window,)) == temporal_split
            
            for i in range(num_lat_splits):
                lat_track = latitude_min + i * lat_split_distance
                
                for j in range(num_long_splits):
                    long_track = longitude_min + j * long_split_distance

                    latitude_mask_burn = (all_burn['LATITUDE'] < lat_track + lat_split_distance) & (all_burn['LATITUDE'] >= lat_track)
                    longitude_mask_burn = (all_burn['LONGITUDE'] < long_track + long_split_distance) & (all_burn['LONGITUDE'] >= long_track)

                    burn_filter_final = burn_filter_year & burn_filter_temporal & latitude_mask_burn & longitude_mask_burn
                    
                    numerical_size = all_burn[burn_filter_final]['FIRE_SIZE_HA'].sum(axis=0)
                    
                    segmented_burn.append(numerical_size)
            
            curr_dir = base_dir + f'/{year}'
            if not os.path.exists(curr_dir):
                os.makedirs(curr_dir)
            
            df = pd.DataFrame(segmented_burn)
            df.to_csv(curr_dir + f'/group_{temporal_split}.csv', index = False, header = False)

In [ ]:
spatially_split_burn(kelowna_burn_data, 6, 6, 2017, 2024, 1, 53, 7)

In [ ]:
check_set = set()

def temporal_process(year_min, year_max, termporal_min, temporal_max, fine_split_min, fine_split_max, window):
    weather_dir = './data/final-weather/'
    base_dir = "./data/weather-stations/kelowna/"
    
    for curr_year in range(year_min,year_max):

        for i in range(fine_split_min,fine_split_max):
            filtered_df = pd.read_csv(base_dir + f"fine_area_{i}_stations.csv")        

            for temporal_split in range(termporal_min,temporal_max):
                temporal_dataframes = []

                for index, row in filtered_df.iterrows():
                    network_name = row['Network Name']
                    native_id = row['Native ID']

                    file_path = f'./data/weather/{curr_year}/{network_name}/{native_id}.csv'
                    temp_df, temporal_data = [], []

                    if os.path.exists(file_path):
                        temp_df = pd.read_csv(file_path, skiprows = 1)
                    else:
                        check_set.add((network_name, native_id))
                        continue

                    if len(temp_df) != 0:
                        temporal_data = temp_df[temp_df[' time'].apply(get_temporal_category, args=(window,)) == temporal_split]
                    
                    if len(temporal_data) != 0:
                        temporal_dataframes.append(temporal_data)
                
#                 print(temporal_dataframes)
                
                if temporal_dataframes:
                    combined_df = pd.concat(temporal_dataframes, ignore_index=True, axis=0, join='outer')
                    combined_df.fillna(-999, inplace = True)
                    if not os.path.exists(weather_dir + f'{curr_year}/group_{temporal_split}'):
                        os.makedirs(weather_dir + f'{curr_year}/group_{temporal_split}')
                    combined_df.to_csv(weather_dir + f'{curr_year}/group_{temporal_split}/fine_area_{i}_weather.csv', index=False, header=True)

In [ ]:
temporal_process(2017, 2022, 1, 53, 1, 37, 7)

In [ ]:
print(len(check_set))

Training Pipeline Begins

In [ ]:
import numpy as np

def align_data():
    curr_year = 2017
    weather_process_base_dir = './data/final-weather/2017'
    new_dir = './data/train-weather/2017'
    
    for group in range(1,13):
        single_dfs = []
        fine_area_weather_missing = set()
        for i in range(1,101):
            file_path = weather_process_base_dir + f'/group_{group}/fine_area_{i}_weather.csv'
            df = []
            if os.path.exists(file_path):
                df = pd.read_csv(file_path)
            else:
                fine_area_weather_missing.add(i)
                continue
            df.replace('None', np.nan, inplace=True)
            
            df = df.apply(pd.to_numeric, errors='coerce')

#             threshold = len(df) * 0.5
            df.replace(-999, np.nan, inplace=True) 

#             cols_to_drop = df.columns[df.isnan().sum() > threshold]
#             df.drop(cols_to_drop, axis=1, inplace=True)

            # impute remaining missing values with column mean
#             df.fillna(df.mean(), inplace=True)

            aggregations = {}
            
            for col in df.columns:
                if pd.api.types.is_numeric_dtype(df[col]):
                    aggregations[col] = 'mean'  
                else:
                    df.drop(col)
            
            df = df.agg(aggregations)
            df = df.to_frame().transpose()
            
            single_dfs.append(df)
        
        if os.path.exists(f'./data/final-burn/{curr_year}/group_{group+1}.csv') and single_dfs:
            combined_df = pd.concat(single_dfs, ignore_index= True)
            threshold = len(combined_df) * 0.8
            cols_to_drop = combined_df.columns[combined_df.isna().sum() > threshold]
#             print(cols_to_drop)
            combined_df.drop(cols_to_drop, axis=1, inplace=True)
            combined_df.fillna(combined_df.mean(), inplace=True)
            combined_df.to_csv(new_dir + f'/group_{group}.csv', index = False)
            
            burn_area_df = pd.read_csv(f'./data/final-burn/{curr_year}/group_{group+1}.csv', header = None, names=['burn_area'])
            for index, row in burn_area_df.iterrows():
                if index+1 in fine_area_weather_missing:
                    burn_area_df.drop(index, inplace = True)
            burn_area_df.to_csv(f'./data/train-burn/{curr_year}/group_{group}.csv', index = False)

In [ ]:
pd.set_option('display.min_rows',70)
pd.set_option('display.max_rows',70)
align_data()

In [ ]:
!pip install xgboost

In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_squared_error

for group in range(15,40):
    

train_df = pd.read_csv('./data/train-weather/2017/group_25.csv')
target_df = pd.read_csv('./data/train-burn/2017/group_25.csv')

X_train = train_df.iloc[:45, :]
y_train = target_df.iloc[:45, 0]
X_test = train_df.iloc[45:60, :]
y_test = target_df.iloc[45:60, 0]

model = xgb.XGBRegressor(objective='reg:squarederror')

model.fit(X_train, y_train)

predictions = model.predict(X_test)

for i in range(len(predictions)):
    print(f"Predicted: {predictions[i]}, Actual: {y_test.iloc[i]}")

mse = mean_squared_error(y_test, predictions)
print("Mean Squared Error on Test Data:", mse)

In [ ]:
# import os
# my_filter = ['ONE_DAY_PRECIPITATION', 'ONE_DAY_RAIN', 'MIN_TEMP', 'ONE_DAY_SNOW', 'time', 'MAX_TEMP']
filtered_df = df
for curr_year in range(2012, 2013):
    yearly_dataframes = []

    for index, row in filtered_df.iterrows():
        network_name = row['Network Name']
        native_id = row['Native ID']
        
        file_path = f'./data/weather/{curr_year}/{network_name}/{native_id}.csv'

        if os.path.exists(file_path):
            temp_df = pd.read_csv(file_path, skiprows=1)
            
            yearly_dataframes.append(temp_df)

    if yearly_dataframes:
        combined_df = pd.concat(yearly_dataframes, ignore_index=True)
        combined_df.columns = [i.lstrip() for i in combined_df.columns]
#         existing_columns = [c for c in my_filter if c in combined_df.columns]
#         combined_df = combined_df[existing_columns]
#       print(combined_df.columns)
        combined_csv_path = f'./training_data_20123/combined_data_{curr_year}.csv'
        combined_df.to_csv(combined_csv_path, index=False)
        print(combined_df)